# 06 — Executed Path Audit (Bug P0 UUIDs)

## Problème

Le benchmark E2E montre `Actual: []` pour presque toutes les traces.  
Le panel a identifié que des **workflow_pattern IDs** (UUIDs) sont stockés comme tool IDs dans `executed_path`.

Ce notebook audite directement la base PostgreSQL pour :
1. Compter les traces avec/sans executed_path
2. Identifier les patterns d'IDs dans executed_path (UUIDs vs FQDN vs normalised)
3. Quantifier l'impact sur les métriques E2E
4. Proposer une stratégie de nettoyage

In [1]:
import subprocess
import json
import re
import os
from collections import Counter, defaultdict
from pathlib import Path

# Load .env file for DB credentials — try multiple paths
env = os.environ.copy()
for candidate in [
    Path(__file__).resolve().parents[2] / ".env" if '__file__' in dir() else None,
    Path.cwd() / "../../.env",
    Path("/home/ubuntu/CascadeProjects/AgentCards/.env"),
]:
    if candidate and candidate.exists():
        for line in candidate.read_text().splitlines():
            if '=' in line and not line.startswith('#'):
                k, v = line.split('=', 1)
                env[k.strip()] = v.strip()
        print(f"Loaded .env from {candidate}")
        break
else:
    print("WARNING: .env not found")

# Parse DATABASE_URL
db_url = env.get('DATABASE_URL', '')
if 'postgres' in db_url:
    import urllib.parse
    parsed = urllib.parse.urlparse(db_url)
    env['PGPASSWORD'] = parsed.password or ''
    PG_USER = parsed.username or 'casys'
    PG_HOST = parsed.hostname or 'localhost'
    PG_PORT = str(parsed.port or 5432)
    PG_DB = parsed.path.lstrip('/') or 'casys'
    print(f"DB: {PG_USER}@{PG_HOST}:{PG_PORT}/{PG_DB}")
else:
    PG_USER, PG_HOST, PG_PORT, PG_DB = 'casys', 'localhost', '5432', 'casys'
    print("WARNING: DATABASE_URL not found, using defaults")

def psql(query):
    """Execute SQL via psql and return rows."""
    result = subprocess.run(
        ['psql', '-h', PG_HOST, '-p', PG_PORT, '-U', PG_USER, '-d', PG_DB, '-t', '-A', '-F', '\t', '-c', query],
        capture_output=True, text=True, env=env
    )
    if result.returncode != 0:
        print(f"ERROR: {result.stderr[:200]}")
        return []
    lines = [l for l in result.stdout.strip().split('\n') if l]
    return lines

def psql_json(query):
    """Execute SQL that returns JSON."""
    result = subprocess.run(
        ['psql', '-h', PG_HOST, '-p', PG_PORT, '-U', PG_USER, '-d', PG_DB, '-t', '-A', '-c', query],
        capture_output=True, text=True, env=env
    )
    if result.returncode != 0:
        print(f"ERROR: {result.stderr[:200]}")
        return None
    raw = result.stdout.strip()
    if not raw:
        return None
    return json.loads(raw)

# Test connection
rows = psql("SELECT count(*) FROM execution_trace")
print(f"Total execution_trace rows: {rows[0] if rows else 'ERROR'}")

Loaded .env from /home/ubuntu/CascadeProjects/AgentCards/.env
DB: casys@localhost:5432/casys
Total execution_trace rows: 1901


## 1. Statut des executed_path

In [2]:
# Count traces with/without executed_path
# Schema: executed_path is text[] (not jsonb), PK is 'id' (not trace_id), no created_at
rows = psql("""
    SELECT 
        count(*) as total,
        count(executed_path) as has_path,
        count(*) - count(executed_path) as no_path,
        count(CASE WHEN array_length(executed_path, 1) IS NULL THEN 1 END) as empty_array,
        count(CASE WHEN array_length(executed_path, 1) > 0 THEN 1 END) as non_empty
    FROM execution_trace
""")
if rows:
    parts = rows[0].split('\t')
    total, has_path, no_path, empty_array, non_empty = [int(x) for x in parts]
    print(f"Total traces:         {total}")
    print(f"Has executed_path:    {has_path} ({has_path/total*100:.1f}%)")
    print(f"  - empty/null array: {empty_array} ({empty_array/total*100:.1f}%)")
    print(f"  - non-empty:        {non_empty} ({non_empty/total*100:.1f}%)")
    print(f"NULL executed_path:   {no_path} ({no_path/total*100:.1f}%)")

Total traces:         1901
Has executed_path:    1901 (100.0%)
  - empty/null array: 53 (2.8%)
  - non-empty:        1848 (97.2%)
NULL executed_path:   0 (0.0%)


In [3]:
# Sample non-empty executed_paths (text[] column, PK=id)
data = psql_json("""
    SELECT json_agg(row_to_json(t)) FROM (
        SELECT id, executed_path, 
               array_length(executed_path, 1) as path_len
        FROM execution_trace 
        WHERE executed_path IS NOT NULL 
          AND array_length(executed_path, 1) > 0
        ORDER BY executed_at DESC
        LIMIT 30
    ) t
""")

if data:
    print(f"Sample of {len(data)} non-empty executed_paths:\n")
    for row in data[:10]:
        path = row['executed_path']
        print(f"  id={str(row['id'])[:12]}... len={row['path_len']}")
        for tool in path[:5]:
            print(f"    → {tool}")
        if len(path) > 5:
            print(f"    ... +{len(path)-5} more")
        print()
else:
    print("No non-empty executed_paths found!")

Sample of 30 non-empty executed_paths:

  id=019c7404-347... len=1
    → syson:syson_diagram_snapshot

  id=019c7403-123... len=1
    → syson:syson_diagram_snapshot

  id=019c7402-fcf... len=1
    → syson:syson_diagram_list

  id=019c73fa-42e... len=1
    → syson:syson_element_children

  id=019c73f7-f2d... len=1
    → syson:syson_element_children

  id=019c73f2-ede... len=1
    → syson:syson_element_children

  id=019c73f2-50c... len=1
    → syson:syson_search

  id=019c73f2-2d8... len=1
    → syson:syson_query_requirements_trace

  id=019c73f0-4ac... len=1
    → onshape:onshape_export_gltf

  id=019c73ef-e3f... len=2
    → pml.mcp.syson.syson_query_aql.6a0c
    → syson:syson_element_children



## 2. Patterns d'IDs dans executed_path

Classifier chaque ID en : UUID, FQDN (`pml.mcp.*`), normalised (`server:tool`), ou inconnu.

In [4]:
UUID_RE = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', re.I)
FQDN_RE = re.compile(r'^pml\.mcp\.')
NORM_RE = re.compile(r'^[a-zA-Z][\w-]*:[a-zA-Z][\w-]*$')
CODE_RE = re.compile(r'^(code|loop):')

def classify_id(tool_id):
    if UUID_RE.match(tool_id):
        return 'UUID'
    if FQDN_RE.match(tool_id):
        return 'FQDN'
    if CODE_RE.match(tool_id):
        return 'code_loop'
    if NORM_RE.match(tool_id):
        return 'normalised'
    return 'unknown'

# Collect all IDs from executed_path (text[] column)
all_paths_data = psql_json("""
    SELECT json_agg(executed_path) 
    FROM execution_trace 
    WHERE executed_path IS NOT NULL 
      AND array_length(executed_path, 1) > 0
""")

all_ids = []
if all_paths_data:
    for path in all_paths_data:
        if isinstance(path, list):
            all_ids.extend(path)
    
    classifications = Counter(classify_id(tid) for tid in all_ids)
    total = len(all_ids)
    
    print(f"Total IDs in all executed_paths: {total}\n")
    print(f"  {'Type':<15s} {'Count':>6s} {'%':>6s}")
    print(f"  {'-'*30}")
    for typ, cnt in classifications.most_common():
        print(f"  {typ:<15s} {cnt:>6d} {cnt/total*100:>5.1f}%")
    
    # Sample each type
    for typ_name, filter_fn in [
        ('UUID', lambda t: UUID_RE.match(t)),
        ('FQDN', lambda t: FQDN_RE.match(t)),
        ('unknown', lambda t: classify_id(t) == 'unknown'),
    ]:
        samples = [tid for tid in all_ids if filter_fn(tid)]
        if samples:
            print(f"\nSample {typ_name} IDs:")
            for s in list(set(samples))[:10]:
                count = samples.count(s)
                print(f"  {s}  (×{count})")
else:
    print("No executed_paths found")

Total IDs in all executed_paths: 3069

  Type             Count      %
  ------------------------------
  normalised        2051  66.8%
  UUID               529  17.2%
  code_loop          282   9.2%
  FQDN               207   6.7%

Sample UUID IDs:
  8401a87d-8133-4257-913e-67b93f84a86c  (×1)
  cf79a32b-a420-4861-bc61-7aef574afd67  (×1)
  b46e991d-2571-4623-bf49-02516e22cb06  (×4)
  c975a4ad-da17-487c-939d-3b4f6fdce5cc  (×1)
  2497f599-c638-4a66-945b-7cf87cdffe90  (×1)
  3abf80cd-e3d5-48db-8346-e4bfeffa17c8  (×3)
  18d318f2-304b-43e4-963b-9a235e1822d1  (×3)
  251b97a4-5459-413a-8d63-0409cc1ee4e3  (×1)
  7992d04f-9296-4aa6-a176-c8bd7c02b4a2  (×1)
  8125f75c-816c-4ed1-a801-a114d322072d  (×14)

Sample FQDN IDs:
  pml.mcp.sim.sim_validate.0175  (×1)
  pml.mcp.onshape.onshape_document_elements.835e  (×5)
  pml.mcp.std.psql_schema.db48  (×2)
  pml.mcp.onshape.onshape_document_list.835e  (×1)
  pml.mcp.syson.syson_search.6a0c  (×2)
  pml.mcp.plm.plm_bom_flatten.0b8b  (×1)
  pml.mcp.syson.sys

## 3. Vérifier si les UUIDs sont des workflow_pattern IDs

In [5]:
# Check if UUIDs exist in workflow_pattern table
uuid_ids_list = [tid for tid in all_ids if UUID_RE.match(tid)] if 'all_ids' in dir() else []

if uuid_ids_list:
    unique_uuids = list(set(uuid_ids_list))[:20]
    uuid_list = "','".join(unique_uuids)
    
    # Check workflow_pattern (PK = pattern_id)
    wp_rows = psql(f"""
        SELECT pattern_id, description, hierarchy_level 
        FROM workflow_pattern 
        WHERE pattern_id IN ('{uuid_list}')
        LIMIT 20
    """)
    
    if wp_rows:
        print(f"UUIDs found in workflow_pattern: {len(wp_rows)}/{len(unique_uuids)}")
        for row in wp_rows[:5]:
            parts = row.split('\t')
            print(f"  {parts[0][:12]}... desc={parts[1][:40]} level={parts[2]}")
    else:
        print("No UUIDs found in workflow_pattern")
    
    # Check tool_schema
    ts_rows = psql(f"""
        SELECT tool_id 
        FROM tool_schema 
        WHERE tool_id IN ('{uuid_list}')
        LIMIT 5
    """)
    
    if ts_rows:
        print(f"\nUUIDs found in tool_schema: {len(ts_rows)} ← these are legit tool IDs")
    else:
        print(f"\nUUIDs found in tool_schema: 0 ← confirmed: NOT tool IDs")
else:
    print("No UUID IDs found in executed_path — good news or no paths at all")

UUIDs found in workflow_pattern: 20/20
  2497f599-c63... desc=Get project statistics: count TypeScript level=0
  cc4c37d9-733... desc=Reload localhost:8081 and take screensho level=0
  18d318f2-304... desc=Generate Lorem Ipsum placeholder text level=0
  8401a87d-813... desc=Test HIL filesystem write capability level=0
  251b97a4-545... desc=Generate fake URL for testing level=0



UUIDs found in tool_schema: 0 ← confirmed: NOT tool IDs


## 4. Impact quantifié sur les données d'entraînement

In [6]:
# Per-trace breakdown: what fraction of each trace's path is UUIDs?
trace_path_data = psql_json("""
    SELECT json_agg(row_to_json(t)) FROM (
        SELECT id, executed_path,
               array_length(executed_path, 1) as path_len
        FROM execution_trace
        WHERE executed_path IS NOT NULL
          AND array_length(executed_path, 1) > 0
    ) t
""")

if trace_path_data:
    stats = {'clean': 0, 'mixed': 0, 'all_uuid': 0, 'all_fqdn': 0, 'other': 0}
    total_tools = 0
    uuid_tools = 0
    fqdn_tools = 0
    
    for row in trace_path_data:
        path = row['executed_path']
        types = [classify_id(tid) for tid in path]
        type_set = set(types)
        total_tools += len(types)
        uuid_tools += types.count('UUID')
        fqdn_tools += types.count('FQDN')
        
        if type_set == {'normalised'} or type_set <= {'normalised', 'code_loop'}:
            stats['clean'] += 1
        elif type_set == {'UUID'}:
            stats['all_uuid'] += 1
        elif type_set == {'FQDN'}:
            stats['all_fqdn'] += 1
        elif 'UUID' in type_set:
            stats['mixed'] += 1
        else:
            stats['other'] += 1
    
    total_traces = sum(stats.values())
    print(f"Traces with non-empty executed_path: {total_traces}")
    print(f"\n  {'Category':<15s} {'Count':>6s} {'%':>6s}")
    print(f"  {'-'*30}")
    for cat, cnt in sorted(stats.items(), key=lambda x: -x[1]):
        print(f"  {cat:<15s} {cnt:>6d} {cnt/total_traces*100:>5.1f}%")
    
    print(f"\nTool IDs in paths:")
    print(f"  Total:     {total_tools}")
    print(f"  UUID:      {uuid_tools} ({uuid_tools/total_tools*100:.1f}%) ← CORRUPTED")
    print(f"  FQDN:      {fqdn_tools} ({fqdn_tools/total_tools*100:.1f}%) ← fixable")
    print(f"  Clean:     {total_tools - uuid_tools - fqdn_tools} ({(total_tools-uuid_tools-fqdn_tools)/total_tools*100:.1f}%)")
else:
    print("No trace path data found")

Traces with non-empty executed_path: 1848

  Category         Count      %
  ------------------------------
  clean             1142  61.8%
  mixed              503  27.2%
  all_fqdn           175   9.5%
  all_uuid            26   1.4%
  other                2   0.1%

Tool IDs in paths:
  Total:     3069
  UUID:      529 (17.2%) ← CORRUPTED
  FQDN:      207 (6.7%) ← fixable
  Clean:     2333 (76.0%)


## 5. Alternative: reconstruire executed_path depuis task_results

Si executed_path est corrompu, on peut peut-être le reconstruire depuis task_results (l'historique des appels MCP).

In [7]:
# Check task_results structure
sample = psql_json("""
    SELECT json_agg(row_to_json(t)) FROM (
        SELECT id, 
               jsonb_array_length(task_results) as n_tasks,
               task_results->0->>'toolName' as first_tool,
               task_results->0->>'serverName' as first_server,
               executed_path,
               array_length(executed_path, 1) as path_len
        FROM execution_trace
        WHERE task_results IS NOT NULL
          AND jsonb_array_length(task_results) > 0
        ORDER BY executed_at DESC
        LIMIT 10
    ) t
""")

if sample:
    print(f"Sample traces with task_results:\n")
    for row in sample:
        path_len = row.get('path_len') or 0
        print(f"  id={str(row['id'])[:12]}... tasks={row['n_tasks']}  path_len={path_len}")
        print(f"    first_tool: {row.get('first_server', '?')}:{row.get('first_tool', '?')}")
        if row.get('executed_path'):
            for tid in row['executed_path'][:3]:
                print(f"    path: {tid}  [{classify_id(tid)}]")
        print()
else:
    print("No traces with task_results found")

Sample traces with task_results:

  id=019c7404-347... tasks=1  path_len=1
    first_tool: None:None
    path: syson:syson_diagram_snapshot  [normalised]

  id=019c7403-123... tasks=1  path_len=1
    first_tool: None:None
    path: syson:syson_diagram_snapshot  [normalised]

  id=019c7402-fcf... tasks=1  path_len=1
    first_tool: None:None
    path: syson:syson_diagram_list  [normalised]

  id=019c73fa-42e... tasks=1  path_len=1
    first_tool: None:None
    path: syson:syson_element_children  [normalised]

  id=019c73f7-f2d... tasks=1  path_len=1
    first_tool: None:None
    path: syson:syson_element_children  [normalised]

  id=019c73f2-ede... tasks=1  path_len=1
    first_tool: None:None
    path: syson:syson_element_children  [normalised]

  id=019c73f2-50c... tasks=1  path_len=1
    first_tool: None:None
    path: syson:syson_search  [normalised]

  id=019c73f2-2d8... tasks=1  path_len=1
    first_tool: None:None
    path: syson:syson_query_requirements_trace  [normalised]

  id

In [8]:
# Can we reconstruct executed_path from task_results?
# task_results is jsonb array of {toolName, serverName, result, durationMs, ...}
# NOTE: serverName/toolName are null in most recent traces (known issue)
reconstruct = psql_json("""
    SELECT json_agg(row_to_json(t)) FROM (
        SELECT 
            id,
            executed_path,
            (
                SELECT json_agg(r.server || ':' || r.tool ORDER BY e.ord)
                FROM jsonb_array_elements(task_results) WITH ORDINALITY AS e(val, ord)
                CROSS JOIN LATERAL (
                    SELECT 
                        COALESCE(e.val->>'serverName', 'unknown') as server,
                        COALESCE(e.val->>'toolName', 'unknown') as tool
                ) r
                WHERE r.server NOT IN ('code', 'loop', 'unknown')
                  AND r.tool != 'unknown'
            ) as reconstructed_path
        FROM execution_trace
        WHERE task_results IS NOT NULL
          AND jsonb_array_length(task_results) > 0
        ORDER BY executed_at DESC
        LIMIT 5
    ) t
""")

if reconstruct:
    print("Path reconstruction from task_results:\n")
    for row in reconstruct:
        orig = row.get('executed_path') or []
        recon = row.get('reconstructed_path') or []
        print(f"  id={str(row['id'])[:12]}...")
        print(f"    original ({len(orig)}):      {orig[:5]}")
        print(f"    reconstructed ({len(recon)}): {recon[:5]}")
        if orig and recon:
            orig_norm = [classify_id(t) for t in orig]
            print(f"    original types: {Counter(orig_norm)}")
            match = orig == recon
            print(f"    exact match: {match}")
        elif not recon:
            print(f"    ⚠ reconstruction empty — serverName/toolName likely null")
        print()
else:
    print("No task_results to reconstruct from")

Path reconstruction from task_results:

  id=019c7404-347...
    original (1):      ['syson:syson_diagram_snapshot']
    reconstructed (0): []
    ⚠ reconstruction empty — serverName/toolName likely null

  id=019c7403-123...
    original (1):      ['syson:syson_diagram_snapshot']
    reconstructed (0): []
    ⚠ reconstruction empty — serverName/toolName likely null

  id=019c7402-fcf...
    original (1):      ['syson:syson_diagram_list']
    reconstructed (0): []
    ⚠ reconstruction empty — serverName/toolName likely null

  id=019c73fa-42e...
    original (1):      ['syson:syson_element_children']
    reconstructed (0): []
    ⚠ reconstruction empty — serverName/toolName likely null

  id=019c73f7-f2d...
    original (1):      ['syson:syson_element_children']
    reconstructed (0): []
    ⚠ reconstruction empty — serverName/toolName likely null



## 6. Fréquence des UUIDs dans executed_path

Certains UUIDs apparaissent beaucoup plus que d'autres — probablement des capabilities très utilisées.

In [9]:
# Top UUIDs by frequency
uuid_ids_for_freq = [tid for tid in all_ids if UUID_RE.match(tid)] if 'all_ids' in dir() else []

if uuid_ids_for_freq:
    uuid_counter = Counter(uuid_ids_for_freq)
    
    print(f"Top 15 UUID IDs in executed_path:\n")
    # Look up in workflow_pattern (PK = pattern_id)
    top_uuids = [uid for uid, _ in uuid_counter.most_common(15)]
    uuid_list = "','".join(top_uuids)
    names = {}
    name_rows = psql(f"""
        SELECT pattern_id, description, hierarchy_level 
        FROM workflow_pattern 
        WHERE pattern_id IN ('{uuid_list}')
    """)
    for row in name_rows:
        parts = row.split('\t')
        names[parts[0]] = (parts[1][:50], parts[2])
    
    print(f"  {'UUID':<40s} {'Count':>5s} {'Description':<50s} {'Level':>5s}")
    print(f"  {'-'*102}")
    for uid, cnt in uuid_counter.most_common(15):
        desc, level = names.get(uid, ('NOT FOUND in workflow_pattern', '?'))
        print(f"  {uid:<40s} {cnt:>5d} {desc[:50]:<50s} L{level:>4s}")
    
    found = sum(1 for uid in top_uuids if uid in names)
    print(f"\n  Resolved: {found}/{len(top_uuids)} UUIDs found in workflow_pattern")
else:
    print("No UUIDs found in executed_path — skipping frequency analysis")

Top 15 UUID IDs in executed_path:

  UUID                                     Count Description                                        Level
  ------------------------------------------------------------------------------------------------------
  0a01917e-79d0-4580-8619-65f5de42ca11       226 Execute SQL queries on PostgreSQL database         L   0
  66c3e406-5220-4bb0-88de-e8fd75b483da        29 Rename capabilities with new namespace, action, de L   0
  6f67f90c-957b-4ccd-a25f-6d28dfcd7680        16 Execute SQL query on embedded PGlite database      L   0
  48ab31f6-5e3e-4370-ab79-4275d7327ef1        15 Rename capability to db:trainingStatus             L   0
  8125f75c-816c-4ed1-a801-a114d322072d        14 Count total capabilities and execution traces in t L   0
  bdbfdf9b-da06-4afe-ae8d-d873ab261017         8 Get git repository status                          L   0
  0b454408-9848-439a-9f49-01cd56e15a08         7 Generate a fake person with their address for test L   0
  5a24b0c2-0

## 7. Stratégie de fix

### Option A: Backfill depuis task_results
- Reconstruire executed_path comme `serverName:toolName` depuis task_results
- Pro: données fiables
- Con: perd l'ordre topologique (task_results est séquentiel d'exécution)

### Option B: Mapper UUID → tool IDs
- Résoudre chaque UUID via workflow_pattern → trouver les tools associés
- Pro: conserve la sémantique originale
- Con: un workflow_pattern ID ne map pas 1:1 vers un tool

### Option C: Filter au benchmark
- Ignorer les traces dont executed_path contient des UUIDs
- Pro: simple, immédiat
- Con: perd des données

### Recommandation: **Option A** (backfill depuis task_results)
- task_results est la source de vérité pour ce qui a réellement été exécuté
- Le format `serverName:toolName` est déjà le format normalisé

In [10]:
# How many traces could be fixed with task_results backfill?
fixable = psql("""
    SELECT 
        count(*) as total,
        count(CASE WHEN task_results IS NOT NULL 
                   AND jsonb_array_length(task_results) > 0 THEN 1 END) as has_task_results,
        count(CASE WHEN (executed_path IS NULL OR array_length(executed_path, 1) IS NULL)
                   AND task_results IS NOT NULL 
                   AND jsonb_array_length(task_results) > 0 THEN 1 END) as fixable_empty,
        count(CASE WHEN executed_path IS NOT NULL 
                   AND array_length(executed_path, 1) > 0 THEN 1 END) as has_path
    FROM execution_trace
    WHERE intent_embedding IS NOT NULL
""")

if fixable:
    parts = fixable[0].split('\t')
    total, has_tr, fix_empty, has_path = [int(x) for x in parts]
    print(f"Traces with intent_embedding (usable for GRU): {total}")
    print(f"  Has task_results:    {has_tr} ({has_tr/total*100:.1f}%)")
    print(f"  Has executed_path:   {has_path} ({has_path/total*100:.1f}%)")
    print(f"  Fixable (empty path, has task_results): {fix_empty} ({fix_empty/total*100:.1f}%)")
    print(f"\n  → Backfill pourrait ajouter {fix_empty} traces avec executed_path")
else:
    print("Query failed")

Traces with intent_embedding (usable for GRU): 1233
  Has task_results:    1233 (100.0%)
  Has executed_path:   1207 (97.9%)
  Fixable (empty path, has task_results): 26 (2.1%)

  → Backfill pourrait ajouter 26 traces avec executed_path


## Conclusions

Ce notebook identifie et quantifie le bug P0 des UUIDs dans executed_path.  
Les résultats informent directement l'**Action 2** du panel Workflow Composer.

### Blocage sur le backfill

Le backfill via `task_results` est **impossible** car `serverName` et `toolName` sont `null`
dans la majorité des traces récentes. La reconstruction donne des paths vides.

### Fix appliqué (2026-02-19)

Le code runtime a été corrigé pour normaliser les FQDN et filtrer les UUIDs/code:/loop: :
- `filterContextTools()` filtre UUIDs, `code:*`, `loop:*`
- `normalizeToolId()` convertit FQDN → `server:action`
- Les **nouvelles traces** sont maintenant propres (format normalisé)

Les données historiques corrompues (17.2% UUID, 6.7% FQDN) ne peuvent pas être backfillées
mais sont filtrées au chargement par le benchmark.